# LC 567: Permutation in String

## 1. Problem

Given two strings `s1` and `s2`, return `True` if `s2` contains a substring that is a permutation of `s1`. Otherwise, return `False`.

A permutation rearranges the characters without changing their frequencies.

The substring must be contiguous, and its length must equal the length of `s1`.

### Example

Input:

`s1 = "ab"`

`s2 = "eidbaooo"`

Output:

`True`

Explanation:

The substring `"ba"` is a permutation of `"ab"` because both contain one `a` and one `b`.

---

## 2. Core Idea

This problem uses **Fixed-Size Sliding Window + Character Frequency Counter**.

A permutation does not require characters to appear in the same order. It requires the same character frequencies.

Therefore, compare the frequency counts of:

- `s1`, which represents the required character frequencies.
- Each window of length `len(s1)` in `s2`.

If the two frequency maps are equal, a permutation exists.

The key condition is:

`window == need`

---

## 3. Pointer / Variable Roles

- `s1`: String whose permutation we want to find.
- `s2`: String in which we search.
- `window_size`: Required length of every candidate substring, equal to `len(s1)`.
- `need`: Frequency counter of characters in `s1`.
- `window`: Frequency counter of characters in the current window of `s2`.
- `right`: Index of the incoming character.
- `s2[right - window_size]`: Character leaving the window.

Important distinction:

- `need` never changes.
- `window` changes as the sliding window moves.
- Both counters must represent character frequencies, not just the characters present.

---

## 4. Initialization

### Step 1: Check the lengths

If:

`len(s1) > len(s2)`

return `False`.

A substring of `s2` cannot be longer than `s2` itself.

### Step 2: Determine the window size

`window_size = len(s1)`

Every candidate substring must have exactly this length.

### Step 3: Build the required frequency map

`need = Counter(s1)`

This stores how many times each character appears in `s1`.

### Step 4: Build the first window

`window = Counter(s2[:window_size])`

This stores the frequencies of the first `window_size` characters of `s2`.

### Step 5: Compare the counters

If:

`need == window`

return `True`.

Otherwise, slide the window.

---

## 5. Main Decision Logic

For every incoming index `right`, starting at `window_size`:

### Step 1: Add the incoming character

`window[s2[right]] += 1`

The new character enters the window.

### Step 2: Remove the outgoing character

`window[s2[right - window_size]] -= 1`

The oldest character leaves the window.

### Step 3: Delete zero-frequency entries

If the outgoing character's frequency becomes zero, remove its key from the counter.

This keeps the counter clean and makes equality comparisons straightforward.

### Step 4: Compare frequency maps

`if window == need:`

If the counters match, the current substring is a permutation of `s1`.

Return `True`.

### Step 5: Finish the search

If no window matches, return `False`.

---

## 6. Why It Works

A permutation preserves the frequency of every character.

For example:

`"aab"` and `"aba"` are permutations because both contain:

- `a`: 2 occurrences
- `b`: 1 occurrence

The algorithm checks every substring of `s2` whose length equals `len(s1)`.

The initial window is checked first. Every subsequent window is produced by adding one character and removing one character.

The frequency counter therefore represents the current window accurately.

If the current counter equals `need`, the window has exactly the same character frequencies as `s1`, so it is a permutation.

Because every possible window of the required length is checked, the algorithm returns the correct result.

---

## 7. Dry Run

### Input

`s1 = "ab"`

`s2 = "eidbaooo"`

Required window size:

`window_size = 2`

Required counter:

`need = {"a": 1, "b": 1}`

Initial window:

`"ei"`

Initial counter:

`window = {"e": 1, "i": 1}`

The counters differ, so continue.

---

## 8. Dry Run Continuation

| Incoming character | Outgoing character | Current window | Frequency match? |
|---|---|---|---|
| `d` | `e` | `"id"` | No |
| `b` | `i` | `"db"` | No |
| `a` | `d` | `"ba"` | Yes |

When the window becomes `"ba"`:

`need = {"a": 1, "b": 1}`

`window = {"b": 1, "a": 1}`

The counters are equal, regardless of key order.

Therefore, return:

`True`

---

## 9. Visual Dry Run

    s1 = "ab"
    s2 = "eidbaooo"
    window_size = 2

    Required frequencies:
    a: 1
    b: 1

    Window: "ei" -> No
    Window: "id" -> No
    Window: "db" -> No
    Window: "ba" -> Yes

    "ba" is a permutation of "ab".

    Answer = True

---

## 10. Code

    class Solution(object):
        from collections import Counter

        def checkInclusion(self, s1, s2):
            if len(s1) > len(s2):
                return False

            window_size = len(s1)

            need = Counter(s1)
            window = Counter(s2[:window_size])

            if need == window:
                return True

            for right in range(window_size, len(s2)):
                window[s2[right]] += 1
                window[s2[right - window_size]] -= 1

                if window[s2[right - window_size]] == 0:
                    del window[s2[right - window_size]]

                if window == need:
                    return True

            return False

---

## 11. Code Breakdown

### Step 1: Handle impossible lengths

If `s1` is longer than `s2`, return `False`.

### Step 2: Calculate the fixed window size

`window_size = len(s1)`

### Step 3: Count the required characters

`need = Counter(s1)`

### Step 4: Count the first window

`window = Counter(s2[:window_size])`

### Step 5: Check the initial window

Compare the counters before entering the sliding-window loop.

### Step 6: Add the incoming character

Increase its frequency in `window`.

### Step 7: Remove the outgoing character

Decrease its frequency and delete its key if the count becomes zero.

### Step 8: Compare the counters

If the frequencies match, return `True`.

### Step 9: Return the fallback result

If every window has been checked without a match, return `False`.

---

## 12. Important Observation

**Permutation checking is a frequency problem, not an ordering problem.**

For example:

`"abc"` and `"bca"` have different orders but identical frequencies.

A direct string comparison would fail:

`"abc" != "bca"`

A frequency comparison succeeds:

`Counter("abc") == Counter("bca")`

This is why a frequency map is more appropriate than comparing the characters position by position.

Also, the window must have exactly `len(s1)` characters. A shorter or longer substring cannot be a permutation of `s1`.

---

## 13. Edge Cases

### Case 1: `s1` is longer than `s2`

`s1 = "abcd"`

`s2 = "ab"`

Answer: `False`

### Case 2: Exact match

`s1 = "abc"`

`s2 = "abc"`

Answer: `True`

### Case 3: Different ordering

`s1 = "abc"`

`s2 = "cba"`

Answer: `True`

### Case 4: Repeated characters

`s1 = "aab"`

`s2 = "eidbaaooo"`

The window `"baa"` has the same frequencies as `"aab"`.

Answer: `True`

### Case 5: No valid permutation

`s1 = "ab"`

`s2 = "eidboaoo"`

No length-two window contains exactly one `a` and one `b`.

Answer: `False`

### Case 6: Single-character pattern

`s1 = "a"`

`s2 = "eidba"`

Answer: `True`

### Case 7: Empty strings

Under the usual problem constraints, both strings are nonempty. If empty strings are allowed, the implementation treats an empty `s1` as having an empty initial window and returns `True`.

---

## 14. Common Mistakes

### Mistake 1: Comparing the strings directly

A permutation can have a different character order.

Compare frequency counters instead.

### Mistake 2: Using a variable-size window

The candidate substring must have exactly `len(s1)` characters.

Use a fixed-size sliding window.

### Mistake 3: Forgetting the initial window

The first window must be checked before the sliding loop because the loop begins with the next incoming character.

### Mistake 4: Forgetting to remove the outgoing character

Without removing the outgoing character, the window grows beyond the required size.

### Mistake 5: Keeping zero-frequency keys

The `Counter` should represent only the characters currently present. Delete a key when its count reaches zero.

### Mistake 6: Comparing only the number of distinct characters

Two windows can contain the same number of distinct characters but different frequencies.

For example, `"aab"` and `"abb"` both have two distinct characters, but their frequency maps differ.

---

## 15. Complexity

Let:

- `n = len(s1)`
- `m = len(s2)`
- `Σ` be the character set size.

### Time Complexity

Building the counters takes `O(n)` and `O(n)` for the initial window.

There are at most `m-n` additional windows. Each window update takes average `O(1)` per dictionary operation, but comparing the counters can take `O(Σ)`.

Overall:

`O(n + (m-n)Σ)`

For lowercase English letters, `Σ = 26`, so the complexity is effectively:

`O(n + m)`

### Space Complexity

`O(Σ)` for the two frequency counters.

For lowercase English letters, this is `O(1)` auxiliary space because the alphabet size is fixed.

---

## 16. Pattern Recognition

**Pattern: Fixed-Size Sliding Window + Frequency Counter Equality**

Recognize this pattern when:

- You need to find a permutation or anagram inside a string.
- The substring length is fixed by another string.
- Character order does not matter.
- Character frequencies must match exactly.

Typical structure:

1. Build the required frequency counter.
2. Build the first window counter.
3. Compare the counters.
4. Slide by adding one character and removing another.
5. Compare after each update.

---

## 17. Pattern Trigger / Generalization

Ask these questions:

1. Does the problem involve a permutation or anagram?
2. Does order not matter, but frequency does?
3. Is the required substring length fixed?
4. Can the frequency state be updated as the window slides?

If yes, consider a fixed-size sliding window with a frequency map.

This approach can be adapted to:

- Finding anagrams in a string.
- Checking whether a permutation exists inside another string.
- Counting substrings with specific character frequencies.
- Matching a multiset of characters.

---

## 18. Pattern Connections

### LC 438: Find All Anagrams in a String

Both problems search for windows whose character frequencies match a pattern string.

- LC 567 returns `True` as soon as a match is found.
- LC 438 returns the starting indices of every matching window.

### LC 2461: Maximum Sum of Distinct Subarrays With Length K

Both use fixed-size sliding windows and maintain information about the current window.

- LC 2461 checks distinctness and tracks a sum.
- LC 567 checks exact frequency equality.

### LC 3: Longest Substring Without Repeating Characters

Both use frequency maps, but their validity conditions differ.

- LC 3 requires every character to appear at most once.
- LC 567 requires the entire frequency map to match `s1`.

### Pattern Hierarchy

    Sliding Window
    |
    |-- Fixed-Size
    |   |-- Sum / Count
    |   |   |-- LC 643: Maximum Average Subarray I
    |   |
    |   |-- Distinctness
    |   |   |-- LC 2461: Maximum Sum of Distinct Subarrays
    |   |
    |   |-- Frequency Equality
    |       |-- LC 567: Permutation in String
    |       |-- LC 438: Find All Anagrams in a String
    |
    |-- Variable-Size
        |-- LC 3: Longest Substring Without Repeating Characters

---

## 19. Interview Explanation

"I use a fixed-size sliding window whose length equals the length of `s1`. I build a frequency counter for `s1` and another for the first window of `s2`. If they match, a permutation exists. Otherwise, I slide the window by adding the incoming character and removing the outgoing character, deleting any zero-frequency entries. After each update, I compare the counters. If no window matches, I return `False`. With a fixed alphabet, the algorithm runs in O(len(s1) + len(s2)) time and O(1) auxiliary space."

---

## 20. Final Summary

The main idea is to find a substring with exactly the same character frequencies as `s1`.

Maintain:

- `need`: Required character frequencies.
- `window`: Current window frequencies.
- `window_size`: Exactly `len(s1)`.

Validity condition:

`window == need`

Complexity for a fixed alphabet:

- Time: `O(len(s1) + len(s2))`
- Auxiliary space: `O(1)`

Mental model:

**A permutation preserves character frequencies. Slide a fixed-size window and compare frequency counters.**

---

## 21. Mastery Checklist + Status

### Checklist

- [x] Identified fixed-size sliding window
- [x] Understood why window size equals `len(s1)`
- [x] Built the required frequency counter
- [x] Built the initial window counter
- [x] Checked the initial window
- [x] Added incoming characters
- [x] Removed outgoing characters
- [x] Deleted zero-frequency keys
- [x] Compared exact character frequencies
- [x] Understood why order does not matter
- [x] Understood time and space complexity
- [x] Connected the problem to anagram-search patterns

### Status

**MASTERED**

### Pattern to Remember

**Fixed-Size Sliding Window + Frequency Counter Equality = Permutation Search**

In [ ]:
class Solution(object):
    from collections import Counter
    def checkInclusion(self, s1, s2):
        """
        :type s1: str
        :type s2: str
        :rtype: bool
        """

        if len(s1)>len(s2):
            return False
        window_size=len(s1)
        
        need=Counter(s1)
        window = Counter(s2[:window_size])
        if need == window:
            return True 
        for right in range(window_size, len(s2)):
            window[s2[right]] += 1
            window[s2[right - window_size]] -= 1
            if window[s2[right - window_size]] == 0:
                del window[s2[right - window_size]]
            if window == need:
                return True        
        return False